# Target selection for the DESI Peculiar Velocity survey
#### **_Authors_**: Christoph Saulder, Kelly Douglass, Cullan Howlett, Khaled Said, Anthony Carr

**NOTE 15th Sep 2026: AC updating for run-1b. Project: https://desi.lbl.gov/desipub/app/PB/show_project?pid=126**

**NOTE 7th May 2021, Christoph noticed that the sweeps and derived files below duplicate SGA objects in the overlap regions between North and South imaging surveys. I have corrected this. Following some more testing, and given the opportunity to update the files one last time, we also modifed one of the FP colour cuts (filter_colour3) by +0.05 mags.**

**NOTE: Updated May 2021, prepared for Main Survey!! Fixed bug in Other SGA selection as this was erroneously removing some galaxies. Following low redshift success rates for pointings at larger radii, set TF objects to only have 2 axial pointings at +/-0.4R(26). Added more frequent pointings for extended galaxies (0.2, 0.4, 0.6, 0.8, 1.0), rather than (0.33, 0.67, 1.0) to increase amount of useful data, but while still providing low priority targeting solutions. Finally, added SGA_ID to final files.**

**NOTE: Updated April 2021, code was applying photo-z and magnitude cuts to SGA galaxies, which is undesirable. Corrected this.**

**NOTE: Updated March 2021, corrected bug in semi-major axis pointings, switched to D(26) for SGA, split into different priority lists, included all SGA objects that aren't FP/TF at low priority. Included hand-picked off-axis targets for very large gals. Duplicated targets in bright and dark time, but note that this is only for SV and will likely not be true later, with split based on SV data**

**NOTE: Updated 18th February 2021, changed to DR9, updated SGA selection and re-arranged the entire selection procedure, moved from composite models to Sersic (because DR9)**

**NOTE: Updated 6th January 2021, to correct for a sign error in the position angle which causes TF and EXT object pointings to be misaligned relative to galaxy imaging, and to use sersic_R (radius) and SGA_pa (position angle) from more up-to-date SGA data**

This file contains code and description for defining the DESI PV targets. The final filenames correspond to 
* **BRIGHT_LOW** - Bright time targets only. Lowest priority. Only intended to provide targets for fibres that would otherwise go to sky.
* **BRIGHT_MEDIUM** - Bright time targets only. Medium priority. Only intended to provide targets for fibres that would otherwise go to sky, but more useful for us than LOW.
* **BRIGHT_HIGH** - Bright time targets only. Highest priority.

For SV purposes, we are duplicating our targets in both bright and dark time, however based on the SV data for later observations we will likely place subsets of galaxies only in one or the other. Hence we are providing 6 separate files to match what we will hopefully be requesting during main survey later.

For our science purposes, the above files contain 6 target types.
* **FPT** - Fundamental Plane Target. Target placed on centre. High priority
* **TFT** - Tully-Fisher Target. Targets placed along semi-major axis at high priority, and on centre at medium priority.
* **KLT** - Kinematic Lensing Target. Targets placed on semi-minor axis to help determine uncertainty in the position angle for the TF sample, and which are also useful for kinematic lensing studies.  Medium priority. *Added in 1b*
* **SGA** - SGA galaxy that is smaller than the DESI fibre patrol radius, but may provide targeting solutions where no other galaxy science objects are available. Target placed on centre at low priority.
* **EXT** - Extended objects that have major axes bigger than DESI fibre patrol radius and so provide targeting solutions where no other galaxy science objects are available. Targets placed along semi-major axis at low priority.
* **EOA** - Extended objects that are bigger than DESI fibre patrol radius in both dimensions and so provide targeting solutions where no other galaxy science objects are available. Targets hand-picked off-axis at low priority.
These are defined first, before then being split over the final six target lists: bright/dark and low/medium/high priority. 

__If you are only interested in the differences between final files, you can stop here.__ 

Note that these target lists are larger than the nominal target density listed in the proposal. This is because during SV we are requesting that Tully-Fisher galaxies are split into more targets (6 per galaxy + 1 in centre) than during main survey (2 per galaxy) and are observed in bright and dark time. This will enable us to test the extent to which the DESI instrument would be able to observe these galaxies and how far along their semi-major axis. During main survey these would only be observed 2 targets per galaxy (+ centre at lower priority).

The first code segment just runs through all the DR8 sweeps in the north and south, and the SGA catalogue, to identify objects of interest and get properties we will need to define the three samples above. The following cuts are applied:
* Z_PHOT_MEDIAN < 0.15 and Z_PHOT_L95 < 0.1. This selects nearby objects that are useful for peculiar velocity surveys. Typical scatter between SGA spec-z and photo-z is 0.02, so these cuts ensure we have as close to complete sample at z<0.1 as possible using photo-z only. See function `redshift_filter`
* The object must be at least one of:
    * The BGS target selection including bitmasks, colour cuts, and r-band fibre_mag cuts. See function `BGS_filter`
    * Or, in the BGS bitmask with magnitude limit r<18.0 and TYPE != PSF. See function `BGS_filter`
    * Or, cross-matched to the SGA catalogue within 1". See function `SGA_filter`

All objects within this selection are output as seperate north and south files from which we can then define the various sub-samples. These sub-samples are tackled in later sections of the notebook because this first stage takes a while (something like 4 hours), and so we have set it up to only (ideally) be run once.

In [2]:
import glob
import numpy as np
import scipy as sp
import astropy.units as u
from astropy.io import fits
from astropy.table import Table, Column, vstack, join
from astropy.coordinates import SkyCoord
from desimodel.footprint import is_point_in_desi
import pandas as pd

In [3]:
#define variables that allow to fine-tune some setting
fp_rband_limit=18.0
fp_sersic_limit=2.5
tf_cosi2_min = np.cos(25.0 * np.pi/180.0)**2
minor_axis_point_ratio = 0.4  # same as major axis, for KL and TF PA estimation
tf_sersic_limit = 2.0

#define datetype for all the info we want to store
galaxydatatype=[("RELEASE",'>i2'),('OBJID','>i4'),('BRICKID','>i4'),('BRICKNAME','S8'),
                ('RA','>f8'),('DEC','>f8'),('TYPE', 'S4'),('SERSIC', 'f'),
                ('FLUX_G','>f4'),('FLUX_R','>f4'),('FLUX_Z','>f4'),
                ('Z_PHOT_MEDIAN','f'),('Z_PHOT_L95','f'),
                ('mag_g','f'),('mag_r','f'),('mag_z','f'),('mag_B','f'),
                ('mag_g_err','f'),('mag_r_err','f'),('mag_z_err','f'),  
                ('fibre_mag_g','f'),('fibre_mag_r','f'),('fibre_mag_z','f'),
                ('uncor_radius','f'),('BA_ratio','f'),('circ_radius','f'),('pos_angle', 'f'),
                ('inSGA','b'),('inBGS','b'),('inlocalbright','b'),('inspecfootprint','b'), 
                ('SGA_pa','f'),('SGA_ba','f'),
                ('SB_D25_g','f'),('SB_D25_r','f'),('SB_D25_z','f'),('RADIUS_SB25','f'),
                ('SGA_MORPHTYPE', 'S21'),('SGA_ID','i8'),('SGA_redshift','f'),('size_SGA','f'),
                ('PMRA', '>f4'),('PMDEC','>f4'),('REF_EPOCH','>f4'),('OVERRIDE','b'),('PVTYPE','>S3'),('PVPRIORITY','>i4'),('POINTINGID','>i4')
                ]

#define datetype for extra information from SGA files
SGA_extra_datatype=[('OBJID','>i4'),
                    ('BRICKID','>i4'),
                    ('BRICKNAME','S8'),
                    ("RELEASE",'>i2'),
                    ('RA','>f8'),
                    ('DEC','>f8'),
                    ('SGA_ID','i8'),
                    ('SGA_pa','f'),
                    ('SGA_ba','f'),
                    ('size_SGA','f'),
                    ('FLUX_G','>f4'),
                    ('FLUX_R','>f4'),
                    ('FLUX_Z','>f4'),
                    ('PMRA', '>f4'),
                    ('PMDEC','>f4'),
                    ('REF_EPOCH','>f4'),
                    # ('mag_B','f'),
                    # ('SB_D25_g','f'),
                    # ('SB_D25_r','f'),
                    # ('SB_D25_z','f'),
                    # ('RADIUS_SB25','f'),
                    # # ('SGA_MORPHTYPE', 'S21'),
                    ('TYPE', 'S4'),
                    ('SERSIC', 'f'),
                    # ('SGA_redshift','f'),
                    ('g-r', '>f4'),
                    ]

#Set the paths to the external sweep files and SGA data
# DR11
pathsouth = '/global/cfs/cdirs/cosmo/data/legacysurvey/dr11/south/sweep/'
pathnorth = '/global/cfs/cdirs/cosmo/data/legacysurvey/dr11/north/sweep/'
fnamesouth = glob.glob(pathsouth+'11.0/*.fits')
fnamenorth = glob.glob(pathnorth+'11.0/*.fits')

# Running through the sweeps takes ages. Set a base directory to save a subset of the data to
savedatapath = '/global/cfs/cdirs/desi/science/td/pv/desi_pv/run_1b/savepath/'

# SGA2025
SGA_hdul = fits.open('/global/cfs/cdirs/cosmo/www/sga/2025/SGA2025-v1.0.fits')
SGA_dataset = SGA_hdul['SGA2025'].data
# SGA_dataset_ellipse = SGA_hdul['ELLIPSEPHOT'].data # Nothing in here is actually needed for the targeting (yay!)
SGA_dataset_tractor = SGA_hdul['TRACTOR'].data
SGA_hdul.close()
# SGA_dataset_ellipse = fits.open("/global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/SGA2025-beta-parent-refcat-v1.6.kd.fits",memmap=True)[1].data
# SGA_dataset_tractor = fits.open('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits',memmap=True)[2].data

In [4]:
#all definition for the basic selection


# Routine to convert from flux to magnitude
def get_mags(flux, fibre_flux, transmission, flux_ivar):
    
    mag = 22.5 - 2.5*np.log10(flux/transmission)
    mag_err = np.abs(2.5/(np.log(10)*flux/transmission*np.sqrt(flux_ivar)))
    fibre_mag = 22.5 - 2.5*np.log10(fibre_flux/transmission)
    
    return mag, mag_err, fibre_mag


# A function to compute radii and other shape information for the galaxies in our first selection
def get_shapes(shape_r,shape_e1,shape_e2):
    
    rad_uncor = shape_r
    help_epsilon = np.sqrt(shape_e1**2 + shape_e2**2)
    BA_ratio = (1.0 - help_epsilon) / (1.0 + help_epsilon)
    rad_circ = rad_uncor * np.sqrt(BA_ratio)
    pos_angle = 0.5 * np.arctan2(shape_e2, shape_e1) / np.pi * 180.0
    
    return rad_uncor,rad_circ,BA_ratio,pos_angle


# get all parameters from SGA file directly, except redshift
def get_SGA_measurements(SGA_dataset, SGA_dataset_tractor):

    SGA_collect = np.zeros(len(SGA_dataset), dtype=SGA_extra_datatype)

    # Tractor info
    SGA_collect['OBJID'] = SGA_dataset_tractor['OBJID']
    SGA_collect['BRICKID'] = SGA_dataset_tractor['BRICKID']
    SGA_collect['BRICKNAME'] = SGA_dataset_tractor['BRICKNAME']
    SGA_collect['RELEASE'] = SGA_dataset_tractor['RELEASE']
    SGA_collect['PMRA'] = SGA_dataset_tractor['PMRA']
    SGA_collect['PMDEC'] = SGA_dataset_tractor['PMDEC']
    SGA_collect['REF_EPOCH'] = SGA_dataset_tractor['REF_EPOCH']
    SGA_collect['TYPE'] = SGA_dataset_tractor['TYPE']
    SGA_collect['SERSIC'] = SGA_dataset_tractor['SERSIC']
    
    # general data. Use SGA-fitted values
    SGA_collect['RA'] = SGA_dataset['RA']
    SGA_collect['DEC'] = SGA_dataset['DEC']
    SGA_collect['FLUX_G'] = SGA_dataset['FLUX_G']
    SGA_collect['FLUX_R'] = SGA_dataset['FLUX_R']
    SGA_collect['FLUX_Z'] = SGA_dataset['FLUX_Z']
    SGA_collect['g-r'] = 2.5*np.log10(SGA_dataset['FLUX_R']/SGA_dataset['FLUX_G'])
    
    # SGA specfic data
    SGA_collect['SGA_ID'] = SGA_dataset['SGAID']
    # SGA_collect['mag_B'] = SGA_dataset_ellipse['mag']
    SGA_collect['size_SGA'] = SGA_dataset['D26'] 
    # SGA_collect['SGA_redshift'] = SGA_dataset_ellipse['Z_LEDA'] 
    # SGA_collect['SGA_MORPHTYPE'] = SGA_dataset_ellipse['MORPHTYPE'] 
    SGA_collect['SGA_pa'] = SGA_dataset['PA']
    SGA_collect['SGA_ba'] = SGA_dataset['BA']
    # SGA_collect['SB_D25_g'] = SGA_dataset_ellipse['G_MAG_SB25'] 
    # SGA_collect['SB_D25_r'] = SGA_dataset_ellipse['R_MAG_SB25'] 
    # SGA_collect['SB_D25_z'] = SGA_dataset_ellipse['Z_MAG_SB25'] 
    # SGA_collect['RADIUS_SB25'] = SGA_dataset_ellipse['RADIUS_SB25']  
    
    return SGA_collect


# A function defining the BGS or other bright objects filter
def BGS_filter(sweepdata, mag_g, mag_r, mag_z, fibremag_r, skymask, basic_cuts,
               redshiftmask=None, 
               ):
    
    # BGS selection    
    help_LS_rr = 22.5 - 2.5*np.log10(sweepdata['FLUX_R'])
    filter_gaia_bright = (sweepdata['GAIA_PHOT_G_MEAN_MAG'] == 0)
    filter_gaia = ((sweepdata['GAIA_PHOT_G_MEAN_MAG'] - help_LS_rr) > 0.6) | filter_gaia_bright
    filter_bgsmask = (sweepdata['MASKBITS'] & (2**1) == 0) & (sweepdata['MASKBITS'] & (2**12) ==0) & (sweepdata['MASKBITS'] & (2**13) == 0)
    filter_fracmasked = (sweepdata['FRACMASKED_G'] < 0.4) & (sweepdata['FRACMASKED_R'] < 0.4) & (sweepdata['FRACMASKED_Z'] < 0.4)
    filter_fracflux = (sweepdata['FRACFLUX_G'] < 5.0) & (sweepdata['FRACFLUX_R'] < 5.0) & (sweepdata['FRACFLUX_Z'] < 5.0)
    filter_fracin = (sweepdata['FRACIN_G'] > 0.3) & (sweepdata['FRACIN_R'] > 0.3) & (sweepdata['FRACIN_Z'] > 0.3)
    
    gmr, rmz = mag_g - mag_r, mag_r - mag_z
    bgs_magcut = (-1 < gmr) & (gmr < 4) & (-1 < rmz) & (rmz < 4)
    bgs_fibremag_cut = ((fibremag_r < (5.1 + mag_r)) & (mag_r <= 17.8)) | ((fibremag_r < 22.9) & (mag_r > 17.8) & (mag_r < 20))

    filter_bright_extended = (mag_r < 18.0) & (sweepdata['TYPE'] != 'PSF ')
    
    if redshiftmask is None:
        allbgsmask = basic_cuts & bgs_magcut & bgs_fibremag_cut & filter_gaia & filter_fracmasked & filter_bgsmask & filter_fracflux & filter_fracin  & skymask
        # Other bright things selection
        allbrightmask = basic_cuts & filter_bright_extended & filter_gaia_bright & filter_bgsmask & filter_fracmasked & skymask
    else:
        allbgsmask = basic_cuts & bgs_magcut & bgs_fibremag_cut & filter_gaia & filter_fracmasked & filter_bgsmask & filter_fracflux & filter_fracin  & skymask & redshiftmask
        # Other bright things selection
        allbrightmask = basic_cuts & filter_bright_extended & filter_gaia_bright & filter_bgsmask & filter_fracmasked & skymask & redshiftmask

    return allbgsmask, allbrightmask


# A function defining the SGA filter
def SGA_filter(sweepdata, coord_sweepdata, mag_g, mag_r, SGA_coords, skymask, basic_cuts):

    # now that SGA and the other photometric data are both DR9, they sould match perfectly ... so only rounding error tolerance (tests with topcat confirm it)
    delta_degree=(0.01/3600.)
        
    # Removes spurious objects close to the SGA centres that are not SGA galaxies to make the cross-matching faster and applies
    # the North/South imaging cut
    onlybrightenough=(mag_g < 20.0) & (mag_r < 20.0) & (sweepdata['TYPE'] != b'PSF ') & skymask & basic_cuts

    nearestneighbor_SGA, d2d, _ = coord_sweepdata[onlybrightenough].match_to_catalog_sky(SGA_coords)  
    SGA_overlap = (d2d < (delta_degree*u.degree))
                
    allsgamask = onlybrightenough
    allsgamask[onlybrightenough] = SGA_overlap
        
    SGA_overlap_indices = np.zeros(len(onlybrightenough), dtype=np.int64)-1
    SGA_overlap_indices[np.where(allsgamask == True)] = nearestneighbor_SGA[SGA_overlap]
    
    SGA_overlap_indices = SGA_overlap_indices[(SGA_overlap_indices>-1)]
    
    return allsgamask, SGA_overlap_indices


# Defines our redshift cuts
def redshift_filter(sweepdata):
    
    # filter high redshift galaxies           
    return (sweepdata['Z_PHOT_MEDIAN'] <= 0.15) & (sweepdata['Z_PHOT_L95'] <= 0.1)
    
    
# Defines a filter for the south
def south_filter(coord_sweepdata):
    
    sc_gal = coord_sweepdata.galactic      
    footprint_south = (sc_gal.b/u.degree <= 0.0) | (coord_sweepdata.dec/u.degree <= 32.375)
    
    return footprint_south


# Defines a filter for the north
def north_filter(coord_sweepdata):
    
    sc_gal = coord_sweepdata.galactic 
    footprint_north = (sc_gal.b/u.degree > 0.0) & (coord_sweepdata.dec/u.degree > 32.375)
    
    return footprint_north


# A function to run over all the sweeps and return the data we care about
def get_sweeps(fname, SGA_collect,
               fname_photoz, 
               area='north'):
    sweepdata = fits.open(fname,memmap=True)[1].data
    photozsweepdata = fits.open(fname_photoz,memmap=True)[1].data
    
    SGA_coords = SkyCoord(ra = SGA_collect['RA']*u.degree, dec = SGA_collect['DEC']*u.degree) 

    # Compute the magnitudes
    mag_g, mag_g_err, fibre_mag_g = get_mags(sweepdata['FLUX_G'], sweepdata['FIBERFLUX_G'], sweepdata['MW_TRANSMISSION_G'], sweepdata['FLUX_IVAR_G'])
    mag_r, mag_r_err, fibre_mag_r = get_mags(sweepdata['FLUX_R'], sweepdata['FIBERFLUX_R'], sweepdata['MW_TRANSMISSION_R'], sweepdata['FLUX_IVAR_R'])
    mag_z, mag_z_err, fibre_mag_z = get_mags(sweepdata['FLUX_Z'], sweepdata['FIBERFLUX_Z'], sweepdata['MW_TRANSMISSION_Z'], sweepdata['FLUX_IVAR_Z'])
    
    # Get the north/south sky cuts
    coord_sweepdata = SkyCoord(ra = sweepdata['RA']*u.degree, dec = sweepdata['DEC']*u.degree) 
    if area.lower() == 'north':
        skymask = north_filter(coord_sweepdata)
    else:
        skymask = south_filter(coord_sweepdata)

    # Some basic filters to remove fluff
    obsfilter = (sweepdata['NOBS_G'] > 0) & (sweepdata['NOBS_R'] > 0) & (sweepdata['NOBS_Z'] > 0)
    filter_minflux = (sweepdata['FLUX_G'] > 0) & (sweepdata['FLUX_R'] > 0) & (sweepdata['FLUX_Z'] > 0)
    basic_cuts = obsfilter &  filter_minflux
    
    # Get the SGA filter
    allsgamask, SGA_overlap_indices = SGA_filter(sweepdata, coord_sweepdata, mag_g, mag_r, SGA_coords, skymask, basic_cuts)

    #nbasic = np.sum(basic_cuts)
    #sweepdata = sweepdata[basic_cuts]
    #photozsweepdata = photozsweepdata[basic_cuts]
    
    #coord_sweepdata = SkyCoord(ra = sweepdata['RA']*u.degree, dec = sweepdata['DEC']*u.degree) 

    #sky_red_filter = redshiftmask & skymask
    
    #sweepdata = sweepdata[sky_red_filter]
    #photozsweepdata = photozsweepdata[sky_red_filter]    

    # Moving redshift filter outside this function
    redshiftmask = None  # redshift_filter(photozsweepdata)
    
    # Compute some shape parameters
    rad_uncor,rad_circ,BA_ratio,pos_angle=get_shapes(sweepdata['SHAPE_R'],sweepdata['SHAPE_E1'],sweepdata['SHAPE_E2'])

    # Get the BGS and bright things filter with a photo-z cut
    allbgsmask, allbrightmask = BGS_filter(sweepdata, mag_g, mag_r, mag_z, fibre_mag_r, skymask, basic_cuts,
                                           redshiftmask)

    # Finally, pull everything together and return objects we want
    full_filter = (allbrightmask | allsgamask | allbgsmask)
    
    nfull = np.sum(full_filter)
    allgal = np.zeros(nfull, dtype = galaxydatatype)

    #transfer general properties from sweep
    for name in ["RELEASE", "OBJID", "BRICKID", "BRICKNAME", "RA", "DEC", "TYPE", "SERSIC", "FLUX_G", "FLUX_R", "FLUX_Z"]:
        allgal[name] = sweepdata[name][full_filter]

    #get photometric redshifts
    for name in ["Z_PHOT_MEDIAN", "Z_PHOT_L95"]:
        allgal[name] = photozsweepdata[name][full_filter]
    
    #transfer precomputed parameters
    names = ["mag_g", "mag_r", "mag_z", "mag_g_err", "mag_r_err", "mag_z_err", "fibre_mag_g", "fibre_mag_r", "fibre_mag_z", "uncor_radius","circ_radius","BA_ratio","pos_angle"]
    for name, vals in zip(names, [mag_g, mag_r, mag_z, mag_g_err, mag_r_err, mag_z_err, fibre_mag_g, fibre_mag_r, fibre_mag_z,rad_uncor,rad_circ,BA_ratio,pos_angle]):
        allgal[name] = vals[full_filter]
        
    #set flags
    allgal['inSGA'] = np.where(allsgamask[full_filter], 1, 0)
    allgal['inBGS'] = np.where(allbgsmask[full_filter], 1, 0)
    allgal['inlocalbright'] = np.where(allbrightmask[full_filter], 1, 0)  
    
    insga=(allgal['inSGA']==1)
    
    # Add SGA info where available. Objects without a cross match have inSGA == 0, SGA_ID = -1, and other properties left empty.
    allgal['SGA_ID'] = -1
    for na in SGA_collect.dtype.names:
         allgal[na][insga]=SGA_collect[na][SGA_overlap_indices]
     
    return allgal    


In [5]:
#run the basic selection

# calculate everything for SGA    
SGA_collect = get_SGA_measurements(SGA_dataset, SGA_dataset_tractor)

# This takes a while (4 hours or so), so by default everything is set to False. 
re_sweep_S = False
re_load_S = False
re_sweep_N = False
re_load_N = False

fullsweep = []
if re_sweep_S:
    for i, fname in enumerate(fnamesouth[:]):
        fname_photoz=pathsouth+'11.0-photo-z/'+fname[-26:-5]+'-pz.fits'
        allgal = get_sweeps(fname, SGA_collect,
                            fname_photoz, area="south")
        print(i, len(fnamesouth), len(allgal))
        np.save(savedatapath+f"sweep_south_{i}", allgal) 

if re_load_S:
    for i, fname in enumerate(fnamesouth[:]):
        allgal = np.load(savedatapath+f"sweep_south_{i}.npy")
        fullsweep.append(allgal)    
    
    fullsweep_south = np.concatenate(fullsweep,axis=0)
    np.save(savedatapath+"fullsweep_south", fullsweep_south)       

# # Then do North
fullsweep = []
if re_sweep_N:
    for i, fname in enumerate(fnamenorth[:]):
        fname_photoz=pathnorth+'11.0-photo-z/'+fname[-26:-5]+'-pz.fits'
        allgal = get_sweeps(fname, SGA_collect, 
                            fname_photoz, area="north")
        print(i, len(fnamenorth), len(allgal))
        np.save(savedatapath+f"sweep_north_{i}", allgal) 

if re_load_N:
    for i, fname in enumerate(fnamenorth[:]):
        allgal = np.load(savedatapath+f"sweep_north_{i}.npy")
        fullsweep.append(allgal)    
    
    fullsweep_north = np.concatenate(fullsweep,axis=0)
    np.save(savedatapath+"fullsweep_north", fullsweep_north)       

/tmp/ipykernel_254832/1386386438.py:48: RuntimeWarning: divide by zero encountered in divide
  SGA_collect['g-r'] = 2.5*np.log10(SGA_dataset['FLUX_R']/SGA_dataset['FLUX_G'])
/tmp/ipykernel_254832/1386386438.py:48: RuntimeWarning: invalid value encountered in divide
  SGA_collect['g-r'] = 2.5*np.log10(SGA_dataset['FLUX_R']/SGA_dataset['FLUX_G'])
/tmp/ipykernel_254832/1386386438.py:48: RuntimeWarning: divide by zero encountered in log10
  SGA_collect['g-r'] = 2.5*np.log10(SGA_dataset['FLUX_R']/SGA_dataset['FLUX_G'])
/tmp/ipykernel_254832/1386386438.py:48: RuntimeWarning: invalid value encountered in log10
  SGA_collect['g-r'] = 2.5*np.log10(SGA_dataset['FLUX_R']/SGA_dataset['FLUX_G'])


In [6]:
def DESI_footprint_filter(objects, desi_tiles, brick_subtract=None):
    
    # Cross match the object coordinates with the DESI tiles
    if brick_subtract is not None:
        bricks_to_subtract = list(set(brick_subtract["brickname"]))
        dr9_overlap = np.isin(objects["BRICKNAME"], bricks_to_subtract)

    survey_overlap = is_point_in_desi(desi_tiles, objects["RA"], objects["DEC"])
    #coord_fullsweep_combi = SkyCoord(ra = fullsweep_combi['RA']*u.degree, dec = fullsweep_combi['DEC']*u.degree) 
    #coord_DESI_tilepointing = SkyCoord(ra = desi_tiles['RA']*u.degree, dec = desi_tiles['DEC']*u.degree) 
    #desi_tile_rad = 1.628 * u.degree

    #nearestneighbor, d2d, _ = coord_fullsweep_combi.match_to_catalog_sky(coord_DESI_tilepointing)  
    #survey_overlap = (d2d < desi_tile_rad)
    
    #everything with grz photometric data north of -30 degree ... based on David's comment
    # survey_overlap=(fullsweep_combi['DEC']>-30.0)

    return survey_overlap & ~dr9_overlap if brick_subtract is not None else survey_overlap

In [7]:
dr9_bricks = fits.open("/global/cfs/cdirs/desi/science/td/pv/desi_pv/run_1b/legacy_dr9_match_all.fits",memmap=True)[1].data

desi_alltiles = Table.read("/global/cfs/cdirs/desi/survey/ops/surveyops/trunk/ops/tiles-main.ecsv")
desi_tiles_1b_bright = desi_alltiles[(desi_alltiles["TILEID"] >= 30993) & (desi_alltiles["TILEID"] <= 33654) & (desi_alltiles["IN_DESI"])]
desi_tiles_1b_dark = desi_alltiles[(desi_alltiles["TILEID"] >= 11962) & (desi_alltiles["TILEID"] <= 15688) & (desi_alltiles["IN_DESI"])]
print(desi_tiles_1b_bright)

TILEID PASS    RA     DEC   ... DESIGNHA DONEFRAC AVAILABLE PRIORITY_BOOSTFAC
                            ...                                              
------ ---- ------- ------- ... -------- -------- --------- -----------------
 30993    0 232.309 -23.444 ...    24.08   0.0000      True             1.000
 30994    0 233.923 -20.461 ...    29.34   0.0000      True             1.000
 30995    0 235.477 -17.464 ...    28.71   0.0000     False             1.000
 30996    0 230.357 -20.639 ...    28.66   0.0000     False             1.000
 30997    0 228.671 -23.589 ...    23.65   0.0000      True             1.000
 30998    0 230.609 -26.428 ...    19.27   1.8771      True             1.000
 30999    0  231.98 -17.673 ...    30.73   0.0000     False             1.000
 31000    0 233.542 -14.691 ...    30.94   0.0000     False             1.000
 31001    0  236.98  -14.45 ...    33.55   0.0000     False             1.000
   ...  ...     ...     ... ...      ...      ...       ...     

## PV_FP - Fundamental Plane Sample

Our FP selection is based on photometrically identifying likely elliptical galaxies using colour cuts and shape parameters. The colour cuts were tuned based on comparisons with visual morphologies from both the SGA and GalaxyZoo. We then simulated the S/N for each target using the fibre magnitude and worked out the magnitude limit above which we were able to recover a high fraction of S/N $> 7.5A^{-1}$ after at most 5x1000s dark time exposures. This S/N limit was chosen based on SV0 data as the point above which we could measure velocity dispersions with less than 10\% relative error. These considerations were then used to define the magnitude limit of the selection, so as to avoid wasting time on targets that would be unlikely to yield good enough S/N.

The selection function for the FP sample is then:
* (TYPE == DEV) OR (TYPE == SER AND (SERSIC > 2.5)) AND (circular radius > 0) AND (ellipticity < 0.7). This keeps elliptical shaped objects well fit by a De Vaucouleurs profile. See function `FP_shape_filter`
* And, colour cuts (g-r > 0.68) AND (g-r > 1.3[r-z]-0.05) AND (g-r < 2.0[r-z]-0.2). This removes objects in the blue cloud/green valley, dusty spirals, and mergers/peculiar objects. See function `FP_colour_filter`
* And, magnitude cut r < 18.0. Beyond this there are generally fewer targets due to our redshift cut, and the velocity dispersion success rate per 0.1 mag bin drops below 50% for a single exposure, and 97.5% after a maximum of 5x1000s dark time exposures. So this is a reasonable sweet spot between capitalising on DESI's abilities (it's a full mag deeper than the SDSS PV survey), but not wasting time on targets or repeating excessively. See function `FP_mag_filter`

We then restrict our selection to objects with Dec > -30.0. See function `DESI_footprint_filter`.

In [8]:
def FP_shape_filter(fullsweep_combi,fp_sersic_limit):

    # Shape filters to check for ellipticals
    filter_rad = (fullsweep_combi['circ_radius'] > 0)
    filter_ellipticity = ((1.0 - fullsweep_combi['BA_ratio']) < 0.7)
    filter_morph = (((fullsweep_combi['TYPE'] == b'DEV ') | (fullsweep_combi['TYPE'] == b'SER')) & (fullsweep_combi['SERSIC'] > fp_sersic_limit))

    return filter_morph & filter_ellipticity & filter_rad

def FP_colour_filter(fullsweep_combi):
    
    #colour cuts to remove blue cloud/green valley, dusty spirals and mergers/peculiar objects
    gmr = fullsweep_combi['mag_g'] - fullsweep_combi['mag_r']
    rmz = fullsweep_combi['mag_r'] - fullsweep_combi['mag_z']
    filter_colour1 = (gmr > 0.68)
    filter_colour2 = (gmr > 1.30*rmz - 0.05)
    filter_colour3 = (gmr < 2.00*rmz - 0.15)
    
    return filter_colour1 & filter_colour2 & filter_colour3
    
def FP_mag_filter(fullsweep_combi,fp_rband_limit):
    
    # Magnitude limit beyond which the simulated velocity dispersion success rate is too low
    return (fullsweep_combi['mag_r'] <= fp_rband_limit)

def inSGA_filter(fullsweep_combi):
    
    # Only objects that are in the SGA.
    return (fullsweep_combi["inSGA"] == 1)

# Sometimes the SGA matching finds multiple profiles at the same coordinates ... but we just need each object once
# this function became obsolete once all input was from the same source (DR9) -CS
#def cleanmulti(fullsweep_combi):
    
#    inSGA = inSGA_filter(fullsweep_combi)
    
    # Identify objects for which the SGA_ID appears more than once
#    multi_sga = np.unique(fullsweep_combi['SGA_ID'][inSGA],return_counts=True)[1]
#    multi_sgamatch = (multi_sga > 1)
    
#    rejectgalaxies = np.zeros(len(fullsweep_combi),dtype=bool)
#    problemsgaidlist = np.unique(fullsweep_combi['SGA_ID'][inSGA])[multi_sgamatch]

    #mark galaxies for rejection
#    counter = 0
#    for multi_id in problemsgaidlist:
#        if counter % 100 == 0:
#            print(counter, len(problemsgaidlist))
#        currentid = (multi_id == fullsweep_combi['SGA_ID'])
#        keepgal = np.argmin(fullsweep_combi['mag_r'][currentid])
#        multis = np.ones(np.sum(currentid),dtype=bool)
#        multis[keepgal] = 0
#        (rejectgalaxies[currentid]) = multis
#        counter += 1
    
    #for the filter, we want to select the galaxies to keep
#    keepgalaxies = np.invert(rejectgalaxies)    
#    return keepgalaxies

In [42]:
# Read in the north and south properties and concatenate them
fullsweep_north = np.load(savedatapath+'fullsweep_north.npy')
fullsweep_south = np.load(savedatapath+'fullsweep_south.npy')
fullsweep_combi = np.concatenate([fullsweep_north, fullsweep_south],axis=0)

# Only keep objects in the (ever changing) DESI spectroscopic footprint
footprint_filter = DESI_footprint_filter(fullsweep_combi, desi_tiles_1b_bright, dr9_bricks)
# low_red_filter = lower_red_filter(fullsweep_combi)
index = np.where(footprint_filter)
fullsweep_combi = fullsweep_combi[index]

#clean data by removing multiple SGA matches ... obsolete with DR9 - CS
#index = cleanmulti(fullsweep_combi)
#fullsweep_combi = fullsweep_combi[index]

# Compute all the FP filters 
mag_filter = FP_mag_filter(fullsweep_combi,fp_rband_limit)
shape_filter = FP_shape_filter(fullsweep_combi,fp_sersic_limit)
colour_filter = FP_colour_filter(fullsweep_combi)
FPredshift_filter = redshift_filter(fullsweep_combi)  # moved redshift cut here for FP only
index = np.where(mag_filter & shape_filter & colour_filter & FPredshift_filter)

# Output the full target list properties for our benefit
filtered_data_fp = fullsweep_combi[index]
ntargets = len(filtered_data_fp)
print(ntargets, len(filtered_data_fp[filtered_data_fp['inSGA'] == 1]))
fits.writeto(savedatapath+"pv_fp_full_bright.fits", filtered_data_fp, overwrite=True)

# Output the reduced target list as required, plus some other information we might need for our purposes
target_data = Table([Column(filtered_data_fp["RELEASE"], name='RELEASE'),
                     Column(filtered_data_fp["OBJID"], name='OBJID'),
                     Column(filtered_data_fp["BRICKID"], name='BRICKID'),
                     Column(filtered_data_fp["BRICKNAME"], name='BRICKNAME'),
                     Column(filtered_data_fp["RA"], name='RA'), 
                     Column(filtered_data_fp["DEC"], name='DEC'), 
                     Column(filtered_data_fp["FLUX_G"], name='FLUX_G'),
                     Column(filtered_data_fp["FLUX_R"], name='FLUX_R'),
                     Column(filtered_data_fp["FLUX_Z"], name='FLUX_Z'),
                     Column(np.zeros(ntargets, dtype='>f4'), name='PMRA'), 
                     Column(np.zeros(ntargets, dtype='>f4'), name='PMDEC'), 
                     Column(np.full(ntargets, 2015.5, dtype='>f4'), name='REF_EPOCH'), 
                     Column(np.full(ntargets, False, dtype='bool'), name='OVERRIDE'), 
                     Column(np.full(ntargets, "FPT", dtype='>S3'), name='PVTYPE'), 
                     Column(np.full(ntargets, 1, dtype='>i4'), name='PVPRIORITY'),
                     Column(np.full(ntargets, 1, dtype='>i4'), name='POINTINGID'),
                     Column(filtered_data_fp["SGA_ID"], name='SGA_ID')])
target_data.write(savedatapath+"pv_fp_bright.fits", format='fits', overwrite=True)

85719 17338


## PV_TF - Tully-Fisher sample

Our TF targets are based on galaxies in SGA. We then apply the inverse of the FP filters, restricting our TF sample to objects with spiral shapes. The following routines are based on Kelly Douglass's code [here](https://github.com/kadglass/DESI_SGA/blob/master/sga_semimajor_ends.ipynb)

The selection function for the TF sample is then:
* inSGA == 1. This means the object is in the SGA catalogue. See function `inSGA_filter` (in previous code block so needs caching)
* (TYPE == EXP)  AND (circular radius > 0) AND (b/a < cos(25.0)). This keeps non-faceon objects well fit by an exponential profile. See function `TF_shape_filter`

We then restrict our selection to objects with Dec >= -30.0. See function `DESI_footprint_filter`.

Out targets for the TF sample are the points at the outer edges of the disk, and the centre at lower priority. The centres of all these targets are expected to already by included in the BGS, and so are put in with OVERRIDE == FALSE to capture this possible overlap. In order to generate our targets, we want to perturb the fibre positions from the centre along the position angle as close to either edge of the disk as possible. These two perturbed pointings are then added to the target list, along with OVERRIDE == TRUE to ensure the fibre is actually placed at the disk edge. See function `TF_disk_edges`. 

However, the exact multiple of R(26) to use is still unknown at the moment and depends on the Halpha flux we can actually get redshift for using redrock. This is currently unclear and so during SV, we are in fact providing targets spaced along the semi-major axis (the same as for our lower priority EXT targets below) at 0.33, 0.67 and 1.0 R(26). See function `grid_positions`. This means the total number of TF targets is larger than the nominal number we asked for during main survey operations, but they can be subsampled down to our requested target density as specified in the DESI Secondary Target proposal description.

**Note: Updated for main survey following SV3 data. Set fibre positions to +/-0.4R(26).**

**Adding in minor axis targets**


In [9]:
def bad_sga(gals):
    '''Identify MW satellites and other objects in the SGA that are not actually galaxies.'''

    bad_sga = Table.read('/global/cfs/cdirs/desi/science/td/pv/desi_pv/run_1b/bad_SGA.txt', 
                         format='ascii.commented_header')

    bad_filter = np.zeros(len(gals), dtype=bool)

    for sgaid in bad_sga['SGAID']:

        bad_filter[gals['SGA_ID'] == sgaid] = True

    return bad_filter


def SGA2020_filter(gals, old_targets):
    '''
    Return just those objects which were part of the Run-1 target lists.
    '''

    sga2020 = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')

    # Find which of the SGA-2020 were originally targeted
    run1_overlap = np.isin(sga2020["SGA_ID"], old_targets['SGA_ID'])

    sga2020_centers = SkyCoord(sga2020['RA'][run1_overlap]*u.deg, sga2020['DEC'][run1_overlap]*u.deg)
    gal_centers = SkyCoord(gals['RA']*u.deg, gals['DEC']*u.deg)

    idx, d2d,_ = gal_centers.match_to_catalog_sky(sga2020_centers)

    # Only keep those objects whose nearest galaxy in the SGA-2020 large galaxy catalog is less than 1" from the center
    old_gals = d2d < 10*u.arcsec
    
    return old_gals


q0 = 0.2

def TF_shape_filter(gals, cosi2_min, sersic_limit):

    # Shape filters to check for inclined spirals
    # filter_rad = (gals['uncor_radius'] > 0)

    # filter_ellipticity = (gals['BA_ratio'] <= tf_ratiomin)
    cosi2 = (gals['SGA_ba']**2 - q0**2)/(1 - q0**2)
    # Galaxies with b/a < q0
    # if cosi2 < 0:
    #     cosi2 = 0
    cosi2 = np.where(cosi2 < 0, 0, cosi2)
    filter_ellipticity = (cosi2 < cosi2_min)

    filter_morph = (gals['TYPE'] == b'EXP ') | ((gals['TYPE'] == b'SER') & (gals['SERSIC'] < sersic_limit))
    #added sersic criterium, because SER is now(DR9) triggered very often -CS
        
    filter_size = (gals["size_SGA"] >= 20.0/60.0)   # There are a few objects that have DIAM < 20" still in SGA so lets remove these for ease.
        
    return filter_morph & filter_ellipticity & filter_size #& filter_rad


def TF_shape_filter_for_minor(gals, cosi2_min, sersic_limit, minor_axis_point_ratio, minimum_sep, ba_limit=0.3):
    """Specifically for finding galaxies that can support minor-axis pointings. Say, whatever min_target_rad > 2 arcsec, so we never override useful targets."""
    
    # Shape filters to check for inclined spirals
    filter_TF = TF_shape_filter(gals, cosi2_min, sersic_limit)

    # Ensure that the minor axis fibers do not overlap with the center fiber
    minor_size = (gals['size_SGA'] * 30.0 * gals['SGA_ba'] * minor_axis_point_ratio) > minimum_sep

    # Let's also add a cut to eliminate super edge-on galaxies
    edge_on = gals['SGA_ba'] > ba_limit
    
    # print(fullsweep_combi['size_SGA'] * 30.0 * fullsweep_combi['SGA_ba'] * minor_axis_point_ratio)
    print(np.sum(filter_TF & minor_size & edge_on))
        
    return filter_TF & minor_size & edge_on


def TF_disk_edges_no_center(gals):
    '''Define target positions 0.4R26 on the major axis.'''
    
    r26 = gals['size_SGA'] / 120.   # arcminutes -> degrees, diameter -> radius
    x = np.array([-0.4, 0.4]).reshape((1,2))
    
    # Distances along the semi-major axis from the center coordinate for our targets
    delta_a = np.dot(r26.reshape((len(r26),1)),x).T
    
    # Target positions
    og_position = SkyCoord(gals['RA']*u.deg, gals['DEC']*u.deg, frame='icrs')
    fiber_pos = og_position.directional_offset_by(gals['SGA_pa']*u.deg, delta_a*u.deg)
    fiber_ra, fiber_dec = fiber_pos.ra.value.T.flatten(), fiber_pos.dec.value.T.flatten()

    fiber_id = np.tile(np.arange(1,3),len(gals))
    pvpriority = np.tile(np.array([1, 1], dtype='>i4'), len(gals))   # Edges at high prority
    override = np.tile(np.array([True, True], dtype='bool'), len(gals)) # Edges have override
    release = np.repeat(gals["RELEASE"], 2)
    objid = np.repeat(gals["OBJID"], 2)
    brickid = np.repeat(gals["BRICKID"], 2)
    brickname = np.repeat(gals["BRICKNAME"], 2)
    flux_g = np.repeat(gals["FLUX_G"], 2)
    flux_r = np.repeat(gals["FLUX_R"], 2)
    flux_z = np.repeat(gals["FLUX_Z"], 2)
    sga_id = np.repeat(gals["SGA_ID"], 2)
    pvtype = np.tile(np.array(["TFT", "TFT"], dtype='>S3'), len(gals))
    pmra = np.repeat(gals['PMRA'], 2)
    pmdec = np.repeat(gals['PMDEC'], 2)
    
    return fiber_ra, fiber_dec, fiber_id, release, objid, brickid, brickname, pvpriority, override, sga_id, flux_g, flux_r, flux_z, pvtype, pmra, pmdec


def TF_disk_edges(gals):
    '''Define target positions on the center and at 0.4R26 on the major axis.'''
    
    r26 = gals['size_SGA'] / 120.   # arcminutes -> degrees, diameter -> radius
    x = np.array([-0.4, 0.0, 0.4]).reshape((1,3))
    
    # Distances along the semi-major axis from the center coordinate for our targets
    delta_a = np.dot(r26.reshape((len(r26),1)),x).T
    
    # Target positions
    og_position = SkyCoord(gals['RA']*u.deg, gals['DEC']*u.deg, frame='icrs')
    fiber_pos = og_position.directional_offset_by(gals['SGA_pa']*u.deg, delta_a*u.deg)
    fiber_ra, fiber_dec = fiber_pos.ra.value.T.flatten(), fiber_pos.dec.value.T.flatten()

    fiber_id = np.tile(np.arange(1,4),len(gals))
    pvpriority = np.tile(np.array([1, 2, 1], dtype='>i4'), len(gals))   # Edges at high prority, centre at low
    override = np.tile(np.array([True, False, True], dtype='bool'), len(gals)) # Edges have override, but not centre.
    release = np.repeat(gals["RELEASE"], 3)
    objid = np.repeat(gals["OBJID"], 3)
    brickid = np.repeat(gals["BRICKID"], 3)
    brickname = np.repeat(gals["BRICKNAME"], 3)
    flux_g = np.repeat(gals["FLUX_G"], 3)
    flux_r = np.repeat(gals["FLUX_R"], 3)
    flux_z = np.repeat(gals["FLUX_Z"], 3)
    sga_id = np.repeat(gals["SGA_ID"], 3)
    pvtype = np.tile(np.array(["TFT", "TFT", "TFT"], dtype='>S3'), len(gals))
    pmra = np.repeat(gals['PMRA'], 3)
    pmdec = np.repeat(gals['PMDEC'], 3)
    
    return fiber_ra, fiber_dec, fiber_id, release, objid, brickid, brickname, pvpriority, override, sga_id, flux_g, flux_r, flux_z, pvtype, pmra, pmdec

# def axis_positions(filtered_data):
    
#     r50 = filtered_data['size_SGA'] / 120.   # arcminutes -> degrees, diameter -> radius
        
#     #x = np.concatenate((np.linspace(-3,-0.5,6), np.linspace(0.5,3,6))).reshape((1,12))
#     x = np.array([-1.0, -0.67, -0.33, 0.0, 0.33, 0.67, 1.0]).reshape((1,7))
    
#     # Distances along the semi-major axis from the center coordinate for our targets
#     delta_a = np.dot(r50.reshape((len(r50),1)),x).T
    
#     # Target positions
#     og_position = SkyCoord(filtered_data['RA']*u.deg, filtered_data['DEC']*u.deg, frame='icrs')
#     fiber_pos = og_position.directional_offset_by(filtered_data['SGA_pa']*u.deg, delta_a*u.deg)
#     fiber_ra, fiber_dec = fiber_pos.ra.value.T.flatten(), fiber_pos.dec.value.T.flatten()

#     #fiber_id = np.tile(np.arange(1,13),len(filtered_data))
#     #objid = np.repeat(filtered_data["OBJID"], 12)
#     #brickid = np.repeat(filtered_data["BRICKID"], 12)
#     #brickname = np.repeat(filtered_data["BRICKNAME"], 12)

#     fiber_id = np.tile(np.arange(1,8),len(filtered_data))
#     pvpriority = np.tile(np.array([1, 1, 1, 2, 1, 1, 1], dtype='>i4'), len(filtered_data))   # Edges at high prority, centre at low
#     override = np.tile(np.array([True, True, True, False, True, True, True], dtype='bool'), len(filtered_data)) # Edges have override, but not centre.
#     objid = np.repeat(filtered_data["OBJID"], 7)
#     brickid = np.repeat(filtered_data["BRICKID"], 7)
#     brickname = np.repeat(filtered_data["BRICKNAME"], 7)
#     sga_id = np.repeat(filtered_data["SGA_ID"], 7)

#     return fiber_ra, fiber_dec, fiber_id, objid, brickid, brickname, pvpriority, override, sga_id

def minor_axis_positions(gals):
    """Recycling above function to add minor-axis positions to SGA galaxies (not just TF)
    Need a relatively large b/a and angular size!"""
    
    r26 = gals['size_SGA'] / 120.   # arcminutes -> degrees, diameter -> radius
    b26 = r26*gals['SGA_ba']  # semi-major -> semi-minor 
        
    y = np.array([-minor_axis_point_ratio, minor_axis_point_ratio]).reshape((1,2))
    
    # Distances along the semi-minor axis from the center coordinate for our targets
    delta_b = np.dot(b26.reshape((len(b26),1)),y).T
    
    # Target positions
    og_position = SkyCoord(gals['RA']*u.deg, gals['DEC']*u.deg, frame='icrs')
    fiber_pos_minor = og_position.directional_offset_by((gals['SGA_pa']-90.)*u.deg, delta_b*u.deg)
    fiber_ra, fiber_dec = fiber_pos_minor.ra.value.T.flatten(), fiber_pos_minor.dec.value.T.flatten()

    fiber_id = np.tile(np.arange(1,3)+3,len(gals))  # +3 because they all have centre and edges already 
    pvpriority = np.tile(np.array([2, 2], dtype='>i4'), len(gals))   # only low priority
    override = np.tile(np.array([True, True], dtype='bool'), len(gals)) # I think they should have override to be useful? Likely won't need to override anything anyway.
    release = np.repeat(gals["RELEASE"], 2)
    objid = np.repeat(gals["OBJID"], 2)
    brickid = np.repeat(gals["BRICKID"], 2)
    brickname = np.repeat(gals["BRICKNAME"], 2)
    flux_g = np.repeat(gals["FLUX_G"], 2)
    flux_r = np.repeat(gals["FLUX_R"], 2)
    flux_z = np.repeat(gals["FLUX_Z"], 2)
    sga_id = np.repeat(gals["SGA_ID"], 2)
    pvtype = np.tile(np.array(["KLT", "KLT"], dtype='>S3'), len(gals))
    pmra = np.repeat(gals['PMRA'], 2)
    pmdec = np.repeat(gals['PMDEC'], 2)

    return fiber_ra, fiber_dec, fiber_id, release, objid, brickid, brickname, pvpriority, override, sga_id, flux_g, flux_r, flux_z, pvtype, pmra, pmdec


In [12]:
# fullsweep_combi = np.concatenate([fullsweep_north, fullsweep_south],axis=0)
# I think we can replace this with just info from the SGA catalog

# Read in the run-1 target list (so we don't duplicate these objects)
run1_tf_targets = Table.read('/global/cfs/cdirs/desi/science/td/pv/desi_pv/savepath_dr9_corr/pv_tf_full.fits')

# Only keep objects in the DESI footprint and SGA.  This means that we want:
# - All SGA-2025 galaxies in the non-DR9 bricks
# - Only those in SGA-2025 that weren't in SGA-2020 in the DR9 bricks
# SGA_only_filter = inSGA_filter(fullsweep_combi)
bright_footprint_filter = DESI_footprint_filter(SGA_collect, desi_tiles_1b_bright)
dark_footprint_filter = DESI_footprint_filter(SGA_collect, desi_tiles_1b_dark)
old_SGA_filter = SGA2020_filter(SGA_collect, run1_tf_targets)
bad_sga_filter = bad_sga(SGA_collect)


################################################################################
# TFT targets in dark tiles 
# (no centers, because the centers were promoted to HIGH in dark time)
#-------------------------------------------------------------------------------
index_only_major = np.where(bright_footprint_filter & dark_footprint_filter & ~old_SGA_filter & ~bad_sga_filter)
only_major_SGA_1b = SGA_collect[index_only_major]

# Compute all the TF filters
shape_filter_only_major = TF_shape_filter(only_major_SGA_1b, tf_cosi2_min, tf_sersic_limit)
index_only_major = np.where(shape_filter_only_major)
filtered_data_only_major = only_major_SGA_1b[index_only_major]

ngalaxies_only_major = len(filtered_data_only_major)

# Define targets
fiber_ra_mjr, fiber_dec_mjr, fiber_id_mjr, release_mjr, objid_mjr, brickid_mjr, brickname_mjr, pvpriority_mjr, override_mjr, sga_id_mjr, flux_g_mjr, flux_r_mjr, flux_z_mjr, pvtype_mjr, pmra_mjr, pmdec_mjr = TF_disk_edges_no_center(filtered_data_only_major)

# To keep the limit below the 20% threshold, demote galaxies with 0 flux in at 
# least one band
# Not quite enough - also remove the 250 reddest objects
demote_sgaid_flux = filtered_data_only_major['SGA_ID'][(filtered_data_only_major['FLUX_G'] <= 0) | (filtered_data_only_major['FLUX_R'] <= 0) | (filtered_data_only_major['FLUX_Z'] <= 0)]
print('number of galaxies with 0 flux in at least one band (g,r,z):', len(demote_sgaid_flux))
filtered_data_only_major_good_flux = filtered_data_only_major[(filtered_data_only_major['FLUX_G'] > 0) & (filtered_data_only_major['FLUX_R'] > 0) & (filtered_data_only_major['FLUX_Z'] > 0)]
filtered_data_only_major_good_flux.sort(order='g-r')
demote_sgaid = np.concatenate([demote_sgaid_flux, filtered_data_only_major_good_flux['SGA_ID'][-125:]])
n = 0
for i in range(len(fiber_ra_mjr)):
    if sga_id_mjr[i] in demote_sgaid:
        n += 1
        pvpriority_mjr[i] += 1
print('Moved', n, 'targets from HIGH to MEDIUM')
################################################################################


################################################################################
# TFT targets in bright tiles only
#-------------------------------------------------------------------------------
index = np.where(bright_footprint_filter & ~dark_footprint_filter & ~old_SGA_filter & ~bad_sga_filter)
new_SGA_1b = SGA_collect[index]

# Compute all the TF filters 
shape_filter = TF_shape_filter(new_SGA_1b, tf_cosi2_min, tf_sersic_limit)
index = np.where(shape_filter)
filtered_data_major = new_SGA_1b[index]
ngalaxies_new = len(filtered_data_major)

# Define targets
fiber_ra, fiber_dec, fiber_id, release, objid, brickid, brickname, pvpriority, override, sga_id, flux_g, flux_r, flux_z, pvtype, pmra, pmdec = TF_disk_edges(filtered_data_major)
################################################################################

fits.writeto(savedatapath+"pv_tf_full_bright.fits", np.concatenate([filtered_data_major, filtered_data_only_major]), overwrite=True)


################################################################################
# KLT targets in bright tiles only
#-------------------------------------------------------------------------------
index_minor = np.where(bright_footprint_filter & ~dark_footprint_filter & ~bad_sga_filter)
all_SGA_1b = SGA_collect[index_minor]

# Compute all the TF filters 
shape_filter_minor = TF_shape_filter_for_minor(all_SGA_1b, tf_cosi2_min, tf_sersic_limit, minor_axis_point_ratio, 2) # DESI fiber diameter = 1.52", so let's round up to 2" minimum separation
index_minor = np.where(shape_filter_minor)

filtered_data_minor = all_SGA_1b[index_minor]
ngalaxies_minor = len(filtered_data_minor)

# Get the target positions by perturbing along the semi-major axis
fiber_ra_mnr,\
fiber_dec_mnr,\
fiber_id_mnr,\
release_mnr,\
objid_mnr,\
brickid_mnr,\
brickname_mnr,\
pvpriority_mnr,\
override_mnr,\
sga_id_mnr,\
flux_g_mnr,\
flux_r_mnr,\
flux_z_mnr,\
pvtype_mnr,\
pmra_mnr,\
pmdec_mnr = minor_axis_positions(filtered_data_minor) 
################################################################################


################################################################################
# KLT targets in dark tiles
# We cannot add all these back in, so we only add them on galaxies with b/a > 
#-------------------------------------------------------------------------------
index_minor_limited = np.where(bright_footprint_filter & dark_footprint_filter & ~bad_sga_filter)
all_SGA_1b_limited = SGA_collect[index_minor_limited]

# Compute all the TF filters 
shape_filter_minor_limited = TF_shape_filter_for_minor(all_SGA_1b_limited, tf_cosi2_min, tf_sersic_limit, minor_axis_point_ratio, 2, ba_limit=0.55) # DESI fiber diameter = 1.52", so let's round up to 2" minimum separation
index_minor_limited = np.where(shape_filter_minor_limited)

filtered_data_minor_limited = all_SGA_1b_limited[index_minor_limited]
ngalaxies_minor_limited = len(filtered_data_minor_limited)

fits.writeto(savedatapath+'pv_kl_full_bright.fits', np.concatenate([filtered_data_minor, filtered_data_minor_limited]), overwrite=True)

# Get the target positions by perturbing along the semi-major axis
fiber_ra_mnr_lim,\
fiber_dec_mnr_lim,\
fiber_id_mnr_lim,\
release_mnr_lim,\
objid_mnr_lim,\
brickid_mnr_lim,\
brickname_mnr_lim,\
pvpriority_mnr_lim,\
override_mnr_lim,\
sga_id_mnr_lim,\
flux_g_mnr_lim,\
flux_r_mnr_lim,\
flux_z_mnr_lim,\
pvtype_mnr_lim,\
pmra_mnr_lim,\
pmdec_mnr_lim = minor_axis_positions(filtered_data_minor_limited) 
################################################################################


ntargets = len(fiber_ra) + len(fiber_ra_mjr) + len(fiber_ra_mnr) + len(fiber_ra_mnr_lim)
print(ngalaxies_new, ngalaxies_only_major, ngalaxies_minor, ngalaxies_minor_limited, ntargets)#, len(filtered_data[filtered_data['inSGA'] == 1]))

# Output the targets
target_data = Table([Column(np.concatenate((objid, objid_mjr, objid_mnr, objid_mnr_lim), axis=None), name='OBJID'),
                     Column(np.concatenate((release, release_mjr, release_mnr, release_mnr_lim), axis=None), name='RELEASE'),
                     Column(np.concatenate((brickid, brickid_mjr, brickid_mnr, brickid_mnr_lim), axis=None), name='BRICKID'),
                     Column(np.concatenate((brickname, brickname_mjr, brickname_mnr, brickname_mnr_lim), axis=None), name='BRICKNAME'),
                     Column(np.concatenate((fiber_ra, fiber_ra_mjr, fiber_ra_mnr, fiber_ra_mnr_lim), axis=None), name='RA'), 
                     Column(np.concatenate((fiber_dec, fiber_dec_mjr, fiber_dec_mnr, fiber_dec_mnr_lim), axis=None), name='DEC'), 
                     Column(np.concatenate((flux_g, flux_g_mjr, flux_g_mnr, flux_g_mnr_lim), axis=None), name='FLUX_G'),
                     Column(np.concatenate((flux_r, flux_r_mjr, flux_r_mnr, flux_r_mnr_lim), axis=None), name='FLUX_R'),
                     Column(np.concatenate((flux_z, flux_z_mjr, flux_z_mnr, flux_z_mnr_lim), axis=None), name='FLUX_Z'),
                     Column(np.concatenate((pmra, pmra_mjr, pmra_mnr, pmra_mnr_lim), axis=None), name='PMRA'), 
                     Column(np.concatenate((pmdec, pmdec_mjr, pmdec_mnr, pmdec_mnr_lim), axis=None), name='PMDEC'), 
                     Column(np.full(ntargets, 2015.5, dtype='>f4'), name='REF_EPOCH'), 
                     Column(np.concatenate((override, override_mjr, override_mnr, override_mnr_lim),axis=None), name='OVERRIDE'), 
                     # Column(np.full(ntargets, "TFT", dtype='>S3'), name='PVTYPE'), 
                     Column(np.concatenate((pvtype, pvtype_mjr, pvtype_mnr, pvtype_mnr_lim), axis=None), name='PVTYPE'), 
                     Column(np.concatenate((pvpriority, pvpriority_mjr, pvpriority_mnr, pvpriority_mnr_lim), axis=None), name='PVPRIORITY'),
                     Column(np.concatenate((fiber_id, fiber_id_mjr, fiber_id_mnr, fiber_id_mnr_lim), axis=None), name='POINTINGID'),
                     Column(np.concatenate((sga_id, sga_id_mjr, sga_id_mnr, sga_id_mnr_lim), axis=None), name='SGA_ID')
                    ])
target_data.write(savedatapath+"pv_tf_bright.fits", format='fits', overwrite=True)

number of galaxies with 0 flux in at least one band (g,r,z): 274
Moved 798 targets from HIGH to MEDIUM
3717
5406
5013 14655 3717 5406 62595


## PV_SGA - Other SGA objects sample

There are a number of relatively small SGA galaxies that don't make it into the FP or TF samples and aren't larger than a fibre patrol radius. However, these are still very easy to target with spare fibres and might be useful for providing targetting solutions during dark time (I think they will already be folded in to bright time). So we add them in just with a single pointing in the centre, with OVERRIDE == FALSE. We also put these at lowest priority, as they are not useful for PV science.

The selection function for the SGA sample is:
* inSGA == 1. This means the object is in the SGA catalogue. See function `inSGA_filter` (in previous code block so needs caching)
* Not FP or TF targets. See function `FP_shape_filter`, `mag_filter`, `colour_filter` and `TF_shape_filter`.
* Smaller than a DESI fibre patrol radius. See function `large_galaxy_filter`.

We then restrict our selection to objects with Dec >= -30.0. See function `DESI_footprint_filter`.

In [45]:
def large_galaxy_filter(gals):
    
    max_patrol_diam = 2.85 # arcminutes
    large_filter = (gals['size_SGA'] >= max_patrol_diam)

    return large_filter

In [47]:
# fullsweep_combi = np.concatenate([fullsweep_north, fullsweep_south],axis=0)

# Read in the run-1 target list (so we don't duplicate these objects)
run1_tf_targets = Table.read('/global/cfs/cdirs/desi/science/td/pv/desi_pv/savepath_dr9_corr/pv_tf_full.fits')
run1_sga_targets = Table.read('/global/cfs/cdirs/desi/science/td/pv/desi_pv/savepath_dr9_corr/pv_sga_full.fits')

# Only keep objects in the DESI footprint and SGA.  This means that we want:
# - All SGA-2025 galaxies in the non-DR9 bricks
# - Only those in SGA-2025 that weren't in SGA-2020 in the DR9 bricks
# SGA_only_filter = inSGA_filter(fullsweep_combi)
footprint_filter = DESI_footprint_filter(SGA_collect, desi_tiles_1b_bright)
old_SGA_filter = SGA2020_filter(SGA_collect, vstack([run1_tf_targets, run1_sga_targets]))
bad_sga_filter = bad_sga(SGA_collect)
# index = np.where(footprint_filter & SGA_only_filter)
index = np.where(footprint_filter & ~old_SGA_filter & ~bad_sga_filter)
# fullsweep_combi = fullsweep_combi[index]
new_SGA_1b = SGA_collect[index]

# checkgal = np.where(fullsweep_combi["SGA_ID"] == 4861001)

# Clean data by removing multiple SGA matches  ..obsolete with DR9
#index = cleanmulti(fullsweep_combi)
#fullsweep_combi = fullsweep_combi[index]

# Keep only galaxies smaller than the patrol radius and which are not in the previous FP or TF selections
fp_coords = SkyCoord(filtered_data_fp['RA']*u.deg, filtered_data_fp['DEC']*u.deg)
sga_coords = SkyCoord(new_SGA_1b['RA']*u.deg, new_SGA_1b['DEC']*u.deg)
idx, d2d,_ = sga_coords.match_to_catalog_sky(fp_coords)
full_filter_FP = (d2d < 1*u.arcsec)
# mag_filter = FP_mag_filter(fullsweep_combi,fp_rband_limit)
# shape_filter_FP = FP_shape_filter(fullsweep_combi,fp_sersic_limit)
# colour_filter = FP_colour_filter(fullsweep_combi)
# full_filter_FP = mag_filter & shape_filter_FP & colour_filter
shape_filter_TF = TF_shape_filter(new_SGA_1b, tf_cosi2_min, tf_sersic_limit)
large_filter = large_galaxy_filter(new_SGA_1b)
# print(large_filter[checkgal], shape_filter_TF[checkgal], shape_filter_FP[checkgal])
index = np.where(~large_filter & ~shape_filter_TF & ~full_filter_FP)

# Output the underlying galaxies 
filtered_data_sga = new_SGA_1b[index]
ntargets_sga = len(filtered_data_sga)
print(ntargets_sga)#, len(filtered_data_sga[filtered_data_sga['inSGA'] == 1]))
fits.writeto(savedatapath+"pv_sga_full_bright.fits", filtered_data_sga, overwrite=True)
                              
# Output the reduced target list as required, plus some other information we might need for our purposes
target_data = Table([Column(filtered_data_sga["OBJID"], name='OBJID'),
                     Column(filtered_data_sga["RELEASE"], name='RELEASE'),
                     Column(filtered_data_sga["BRICKID"], name='BRICKID'),
                     Column(filtered_data_sga["BRICKNAME"], name='BRICKNAME'),
                     Column(filtered_data_sga["RA"], name='RA'), 
                     Column(filtered_data_sga["DEC"], name='DEC'), 
                     Column(filtered_data_sga["FLUX_G"], name='FLUX_G'),
                     Column(filtered_data_sga["FLUX_R"], name='FLUX_R'),
                     Column(filtered_data_sga["FLUX_Z"], name='FLUX_Z'),
                     Column(filtered_data_sga["PMRA"], name='PMRA'), 
                     Column(filtered_data_sga["PMDEC"], name='PMDEC'), 
                     Column(np.full(ntargets_sga, 2015.5, dtype='>f4'), name='REF_EPOCH'), 
                     Column(np.full(ntargets_sga, False, dtype='bool'), name='OVERRIDE'), 
                     Column(np.full(ntargets_sga, "SGA", dtype='>S3'), name='PVTYPE'), 
                     Column(np.full(ntargets_sga, 3, dtype='>i4'), name='PVPRIORITY'),
                     Column(np.full(ntargets_sga, 1, dtype='>i4'), name='POINTINGID'),
                     Column(filtered_data_sga["SGA_ID"], name='SGA_ID')])
target_data.write(savedatapath+"pv_sga_bright.fits", format='fits', overwrite=True)

17283


## PV_EXT - Extended objects sample

Our EXT targets are based on galaxies in SGA what are larger than the fibre patrol radius. As such, they are ideal for provide a targetting solution for a fibre that would otherwise have no science target. For these, we provide a grid of targets spaced along the semi-major axis. The following routines are based on Kelly Douglass's code [here](https://github.com/kadglass/DESI_SGA/blob/master/sga_semimajor_grids.ipynb)

The selection function for the EXT sample is:
* inSGA == 1. This means the object is in the SGA catalogue. See function `inSGA_filter` (in previous code block so needs caching)
* Large galaxies with D(26) >= 2.85 arcmins. See function `large_galaxy_filter`

We then restrict our selection to objects with Dec >= -30.0. See function `DESI_footprint_filter`.

We then treat the galaxies differently depending on whether they already made it into the FP or TF samples previously (See functions `FP_shape_filter` and `TF_shape_filter`). If the galaxy is already FP, we add semi-major axis pointings at 0.33, 0.67 and 1.0 times D(26) at lowest priority and with OVERRIDE == TRUE to provide some more targetting solutions if the fibre falls on this galaxy. If the galaxy is already TF, for SV we are doing nothing - these already have semi-major axis pointings at the same factors of D(26) as above. In the main survey, we would add some additional low priority semi-major axis targets with OVERRIDE == TRUE. Finally, if the galaxy is neither FP or TF, we add a central pointing with OVERRIDE == FALSE, and the six semi-major axis pointings with OVERRIDE == TRUE, all again at lowest priority. See function `grid_positions`.

Note that we also provide hand-picked off-axis pointings for these very large galaxies as they could also be very wide, such that semi-major axis pointings alone don't provide enough targets within the SGA ellipse.

Note, given low redshift success rate from SV3 past 0.4R(26), I upped the number of pointings to include values closer to the centre, while still providing targets at larger radii for targetting solutions. New values are +/-(0.2, 0.4, 0.6, 0.8, 1.0) R(26).

**Update (KAD, Sept. 19, 2026): Due to the 20% limit we have for increasing the target lists for run-1b, we are removing all EXT targets from the bright lists.**

In [48]:
def grid_positions(gals, target_type="other"):
    
    r26 = gals['size_SGA'] / 120.   # arcminutes -> degrees, diameter -> radius
        
    # Centre already included in FP at priority 1, other pointings added at priority 3.
    # Let's also add the KLT targets to these galaxies, at priority 3
    if target_type == "fp":
        x = np.array([-1.0, -0.8, -0.6, -0.4, -0.2, 0.2, 0.4, 0.6, 0.8, 1.0]).reshape((1,10))
        y = np.array([-0.4, 0.4]).reshape((1,2))
        priority = np.array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], dtype='>i4')
        place = np.array([True, True, True, True, True, True, True, True, True, True, True, True], dtype='bool')
        pv_type = np.array(10*['EXT'] + 2*['KLT'], dtype='>S3')
        
    # Values at 0.4 already included in TF at priority 1, and centre at priority 2. 
    # Let's add other pointings at priority 2 as they might be useful for PV.
    elif target_type == "tf":
        x = np.array([-1.0, -0.8, -0.6, -0.2, 0.2, 0.6, 0.8, 1.0]).reshape((1,8))
        y = None
        priority = np.array([2, 2, 2, 2, 2, 2, 2, 2], dtype='>i4')
        place = np.array([True, True, True, True, True, True, True, True], dtype='bool')
        pv_type = np.array(8*['EXT'], dtype='>S3')
        
    # Not of interest for FP or TF. Add all pointings at priority 3
    # Let's also add the KLT targets to these galaxies, at priority 3
    else:
        x = np.array([-1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0]).reshape((1,11))
        y = np.array([-0.4, 0.4]).reshape((1,2))
        priority = np.array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], dtype='>i4')
        place = np.array([True, True, True, True, True, False, True, True, True, True, True, True, True], dtype='bool')
        pv_type = np.array(11*['EXT'] + 2*['KLT'], dtype='>S3')
    
    # Distances along the semi-major axis from the center coordinate for our targets
    delta_a = np.dot(r26.reshape((len(r26),1)),x).T
    
    # Target positions on major axis
    og_position = SkyCoord(gals['RA']*u.deg, gals['DEC']*u.deg, frame='icrs')
    fiber_pos = og_position.directional_offset_by(gals['SGA_pa']*u.deg, delta_a*u.deg)
    fiber_ra, fiber_dec = fiber_pos.ra.value.T.flatten(), fiber_pos.dec.value.T.flatten()

    # Adding targets on the minor axis
    if y is not None:
        # Distance along the semi-minor axis from the center coordinate for our targets
        delta_b = np.dot(r26.reshape((len(r26), 1)), y).T

        # Target positions on minor axis
        fiber_pos_minor = og_position.directional_offset_by(gals['SGA_pa']*u.deg - 90*u.deg, delta_b*u.deg)

        # Flatten and concatenate to major axis positions
        fiber_ra_minor, fiber_dec_minor = fiber_pos_minor.ra.value.T.flatten(), fiber_pos_minor.dec.value.T.flatten()
        fiber_ra = np.concatenate([fiber_ra, fiber_ra_minor])
        fiber_dec = np.concatenate([fiber_dec, fiber_dec_minor])

    nx = len(place)
    fiber_id = np.tile(np.arange(1, nx+1),len(gals))
    pvpriority = np.tile(priority, len(gals))   # Edges at high prority, centre at low
    override = np.tile(place, len(gals)) # Edges have override, but not centre.
    pvtype = np.tile(pv_type, len(gals))
    release = np.repeat(gals["RELEASE"], nx)
    objid = np.repeat(gals["OBJID"], nx)
    brickid = np.repeat(gals["BRICKID"], nx)
    brickname = np.repeat(gals["BRICKNAME"], nx)
    flux_g = np.repeat(gals["FLUX_G"], nx)
    flux_r = np.repeat(gals["FLUX_R"], nx)
    flux_z = np.repeat(gals["FLUX_Z"], nx)
    sga_id = np.repeat(gals["SGA_ID"], nx)
    pmra = np.repeat(gals['PMRA'], nx)
    pmdec = np.repeat(gals['PMDEC'], nx)

    return fiber_ra, fiber_dec, fiber_id, release, objid, brickid, brickname, pvpriority, override, sga_id, flux_g, flux_r, flux_z, pmra, pmdec, pvtype

In [49]:
# fullsweep_combi = np.concatenate([fullsweep_north, fullsweep_south],axis=0)

# Read in the run-1 target list (so we don't duplicate these objects)
run1_tf_targets = Table.read('/global/cfs/cdirs/desi/science/td/pv/desi_pv/savepath_dr9_corr/pv_tf_full.fits')
run1_ext_targets = Table.read('/global/cfs/cdirs/desi/science/td/pv/desi_pv/savepath_dr9_corr/pv_ext_full.fits')

# Only keep objects in the DESI footprint and SGA.  This means that we want:
# - All SGA-2025 galaxies in the non-DR9 bricks
# - Only those in SGA-2025 that weren't in SGA-2020 in the DR9 bricks
# SGA_only_filter = inSGA_filter(fullsweep_combi)
footprint_filter = DESI_footprint_filter(SGA_collect, desi_tiles_1b_bright)
old_SGA_filter = SGA2020_filter(SGA_collect, vstack([run1_tf_targets, run1_ext_targets]))
bad_sga_filter = bad_sga(SGA_collect)
# index = np.where(footprint_filter & SGA_only_filter)
index = np.where(footprint_filter & ~old_SGA_filter & ~bad_sga_filter)
# fullsweep_combi = fullsweep_combi[index]
new_SGA_1b = SGA_collect[index]

# Clean data by removing multiple SGA matches  ..obsolete with DR9
#index = cleanmulti(fullsweep_combi)
#fullsweep_combi = fullsweep_combi[index]

# Keep only galaxies larger than the patrol radius
fp_coords = SkyCoord(filtered_data_fp['RA']*u.deg, filtered_data_fp['DEC']*u.deg)
sga_coords = SkyCoord(new_SGA_1b['RA']*u.deg, new_SGA_1b['DEC']*u.deg)
idx, d2d,_ = sga_coords.match_to_catalog_sky(fp_coords)
full_filter_FP = (d2d < 0.1*u.arcsec)
# mag_filter = FP_mag_filter(fullsweep_combi,fp_rband_limit)
# shape_filter_FP = FP_shape_filter(fullsweep_combi,fp_sersic_limit)
# colour_filter = FP_colour_filter(fullsweep_combi)
shape_filter_TF = TF_shape_filter(new_SGA_1b, tf_cosi2_min, tf_sersic_limit)
large_filter = large_galaxy_filter(new_SGA_1b)
index_large = np.where(large_filter)
index_FP = np.where(large_filter & full_filter_FP) #& shape_filter_FP & mag_filter & colour_filter)
index_TF = np.where(large_filter & shape_filter_TF)
index_other = np.where(large_filter & ~full_filter_FP & ~shape_filter_TF)

# Output the underlying galaxies 
filtered_data_large = new_SGA_1b[index_large]
ngalaxies_large = len(filtered_data_large)
fits.writeto(savedatapath+"pv_ext_full_bright.fits", filtered_data_large, overwrite=True)

# Get the target positions by perturbing along the semi-major axis. We need to do 
# this differently if the EXT galaxy already made it into the FP or TF selections 
# to avoid duplicating targets, and get the correct priorities.
filtered_data_FP = new_SGA_1b[index_FP]
filtered_data_TF = new_SGA_1b[index_TF]
filtered_data_other = new_SGA_1b[index_other]

data_FP = grid_positions(filtered_data_FP, target_type="fp")
data_TF = grid_positions(filtered_data_TF, target_type="tf")
data_other = grid_positions(filtered_data_other, target_type="other")

fiber_ra = np.concatenate([data_FP[0], data_TF[0], data_other[0]])
fiber_dec = np.concatenate([data_FP[1], data_TF[1], data_other[1]])
fiber_id = np.concatenate([data_FP[2], data_TF[2], data_other[2]])
release = np.concatenate([data_FP[3], data_TF[3], data_other[3]])
objid = np.concatenate([data_FP[4], data_TF[4], data_other[4]])
brickid = np.concatenate([data_FP[5], data_TF[5], data_other[5]])
brickname = np.concatenate([data_FP[6], data_TF[6], data_other[6]])
pvpriority = np.concatenate([data_FP[7], data_TF[7], data_other[7]])
override = np.concatenate([data_FP[8], data_TF[8], data_other[8]])
sga_id = np.concatenate([data_FP[9], data_TF[9], data_other[9]])
flux_g = np.concatenate([data_FP[10], data_TF[10], data_other[10]])
flux_r = np.concatenate([data_FP[11], data_TF[11], data_other[11]])
flux_z = np.concatenate([data_FP[12], data_TF[12], data_other[12]])
pmra = np.concatenate([data_FP[13], data_TF[13], data_other[13]])
pmdec = np.concatenate([data_FP[14], data_TF[14], data_other[14]])
pv_type = np.concatenate([data_FP[15], data_TF[15], data_other[15]])

# Target stats
ntargets_large = len(fiber_ra)
print(ngalaxies_large, ntargets_large)#, len(filtered_data[filtered_data['inSGA'] == 1]))
print('FP:', len(filtered_data_FP), len(data_FP[0]))
print('TF:', len(filtered_data_TF), len(data_TF[0]))
print('other:', len(filtered_data_other), len(data_other[0]))
    
# Output the targets
target_data = Table([Column(release, name='RELEASE'),
                     Column(objid, name='OBJID'),
                     Column(brickid, name='BRICKID'),
                     Column(brickname, name='BRICKNAME'),
                     Column(fiber_ra, name='RA'), 
                     Column(fiber_dec, name='DEC'), 
                     Column(flux_g, name='FLUX_G'),
                     Column(flux_r, name='FLUX_R'),
                     Column(flux_z, name='FLUX_Z'),
                     Column(pmra, name='PMRA'), 
                     Column(pmdec, name='PMDEC'), 
                     Column(np.full(ntargets_large, 2015.5, dtype='>f4'), name='REF_EPOCH'), 
                     Column(override, name='OVERRIDE'), 
                     Column(pv_type, name='PVTYPE'), 
                     Column(pvpriority, name='PVPRIORITY'),
                     Column(fiber_id, name='POINTINGID'),
                     Column(sga_id, name='SGA_ID')])
target_data.write(savedatapath+"pv_ext_bright.fits", format='fits', overwrite=True)

484 5449
FP: 168 2016
TF: 135 1080
other: 181 2353


## PV_EOA - Extra off-axis targets

We now have lists of FP, TF, SGA and EXT targets from the above codes. We also have a set of hand-picked targets for very large SGA galaxies, denoted as EOA from here on. Given these the last thing to do is recombine them based on their relative priority. The split is based on PV_PRIORITY, and targets are duplicated in bright and dark time:

**Randomising is not currently working if adding in minor axis pointings because not all TF galaxies have the same number of targets**

**Update (KAD, Sept. 19, 2026): Due to the 20% limit we have for increasing the target lists for run-1b, we are removing all EOA targets from the bright lists.**

In [40]:
# np.random.seed(2011)

# Read in the pv_fp, pv_tf, pv_sga, pv_ext and pv_eoa files
# pv_fp = fits.open(savedatapath+'pv_fp_bright.fits', memmap=True)[1].data
# pv_tf = fits.open(savedatapath+'pv_tf_bright.fits', memmap=True)[1].data
# pv_sga = fits.open(savedatapath+'pv_sga_bright.fits', memmap=True)[1].data
# pv_ext = fits.open(savedatapath+'pv_ext_bright.fits', memmap=True)[1].data
pv_eoa = Table.read(savedatapath+'SGA2025_off-axis_targets.fits')

# Convert SGA ID column in pv_eoa table to int
pv_eoa['SGA_ID'] = pv_eoa['SGAID'].astype(int)

# Filter the EOA targets to keep only those in the run-1b BRIGHT tiles
footprint_filter = DESI_footprint_filter(pv_eoa, desi_tiles_1b_bright)
index_eoa = np.where(footprint_filter)
filtered_data_eoa = pv_eoa[index_eoa]

# fullsweep_combi = np.concatenate([fullsweep_north, fullsweep_south],axis=0)
# SGA_only_filter = inSGA_filter(fullsweep_combi)
# footprint_filter_nobricksubtract = DESI_footprint_filter(fullsweep_combi, desi_tiles_1b_bright)
# index = np.where(footprint_filter_nobricksubtract)
# fullsweep_combi = fullsweep_combi[index]

# # Randomise these into pv_bright and pv_dark lists. First do the TF targets. Keep galaxies with multiple
# # pointings consecutive in case list gets truncated or something.
# npointings_tf = np.amax(pv_tf["POINTINGID"])
# pv_tf_sortid = npointings_tf * np.random.permutation(len(pv_tf[pv_tf["POINTINGID"] == 1]))
# pv_tf_idlist = np.concatenate([range(i,i+npointings_tf) for i in pv_tf_sortid])
# pv_tf_randomised = pv_tf[pv_tf_idlist]

# # Now the EXT targets. Let's just randomly reorder these as they have different numbers of pointings
# # and truncation is less important.
# pv_ext_sortid = np.random.permutation(len(pv_ext))
# pv_ext_randomised = pv_ext[pv_ext_sortid]

# # Now the SGA targets
# pv_sga_sortid = np.random.permutation(len(pv_sga))
# pv_sga_randomised = pv_sga[pv_sga_sortid]

# Finally the Extended Off-Axis (EOA) targets for very large galaxies. 
# These are just a list of RAs and Decs, so let's create the supplementary information here. 
# They are in SGA, so they will adopt their SGA galaxy properties
filtered_data_eoa_info = join(filtered_data_eoa, SGA_collect, keys='SGA_ID', join_type='left')
# SGA_lookup = pd.DataFrame(fullsweep_combi).drop_duplicates(subset="SGA_ID")
# EOA_DF = pd.DataFrame(pv_eoa).rename(columns={"SGAID": "SGA_ID"})
# EOA_DF = EOA_DF.astype(dtype={"RA":"=f8", "DEC":"=f8", "SGA_ID":"=i8"})
# EOA_DF = EOA_DF[EOA_DF["SGA_ID"].isin(SGA_lookup["SGA_ID"])]
# SGA_IDs = EOA_DF["SGA_ID"].astype(np.int64)
# for col in ["RELEASE", "OBJID", "BRICKID", "BRICKNAME", "FLUX_G", "FLUX_R", "FLUX_Z"]:
#     EOA_DF[col] = EOA_DF["SGA_ID"].astype(np.int64).map(SGA_lookup.set_index("SGA_ID")[col])
# fibre_ids = []
# done = []
# for SGA_ID in SGA_IDs:
#     if SGA_ID not in done:
#         print(SGA_ID, np.sum(SGA_lookup["SGA_ID"].astype(np.int64)==SGA_ID), np.sum(EOA_DF["SGA_ID"].astype(np.int64)==SGA_ID))
#         no_targets = np.sum(EOA_DF["SGA_ID"].astype(np.int64)==SGA_ID)
#         fibre_ids = [*fibre_ids, *np.arange(no_targets)]  # too difficult to do this with respect to the other pointings already there
#         done.append(SGA_ID)

neoa = len(filtered_data_eoa_info)
print(neoa)
# ntargets = len(pv_fp) + len(pv_tf) + len(pv_sga) + len(pv_ext) + neoa
# print(ntargets)

pv_eoa_formatted = np.zeros(neoa, dtype=galaxydatatype)
pv_eoa_formatted["RA"] = filtered_data_eoa_info['RA_1'] #EOA_DF["RA"].astype(">f8")
pv_eoa_formatted["DEC"] = filtered_data_eoa_info['DEC_1'] #EOA_DF["DEC"].astype(">f8")
pv_eoa_formatted["OBJID"] = filtered_data_eoa_info['OBJID'] #EOA_DF["OBJID"].astype(">i4")
pv_eoa_formatted["RELEASE"] = filtered_data_eoa_info['RELEASE'] #EOA_DF["RELEASE"].astype(">i2")
pv_eoa_formatted["BRICKID"] = filtered_data_eoa_info['BRICKID'] #EOA_DF["BRICKID"].astype(">i4")
pv_eoa_formatted["BRICKNAME"] = filtered_data_eoa_info['BRICKNAME'] #EOA_DF["BRICKNAME"].astype(">S8")
pv_eoa_formatted["FLUX_G"] = filtered_data_eoa_info['FLUX_G'] #EOA_DF["FLUX_G"].astype(">f4")
pv_eoa_formatted["FLUX_R"] = filtered_data_eoa_info['FLUX_R'] #EOA_DF["FLUX_R"].astype(">f4")
pv_eoa_formatted["FLUX_Z"] = filtered_data_eoa_info['FLUX_Z'] #EOA_DF["FLUX_Z"].astype(">f4")
pv_eoa_formatted["PMRA"] = filtered_data_eoa_info['PMRA'] #np.zeros(neoa, dtype='>f4')
pv_eoa_formatted["PMDEC"] = filtered_data_eoa_info['PMDEC'] #np.zeros(neoa, dtype='>f4')
pv_eoa_formatted["REF_EPOCH"] = np.full(neoa, 2015.5, dtype='>f4')
pv_eoa_formatted["OVERRIDE"] = np.full(neoa, True, dtype='bool')
pv_eoa_formatted["PVTYPE"] = np.full(neoa, "EOA", dtype='>S3')
pv_eoa_formatted["PVPRIORITY"] = np.full(neoa, 3, dtype='>i4')
pv_eoa_formatted["POINTINGID"] = np.arange(13, neoa+13, dtype='>i4') #np.array(fibre_ids, dtype='>i4')  
pv_eoa_formatted["SGA_ID"] = filtered_data_eoa_info['SGA_ID'] #EOA_DF["SGA_ID"].astype("i8")

2629


## PV calibrators

Before inserting EOA targets, load in calibrator list, subtract any targets already on those objects, then add everything together with cals at the front

In [50]:
pv_fp = fits.open(savedatapath+'pv_fp_bright.fits', memmap=True)[1].data
pv_tf = fits.open(savedatapath+'pv_tf_bright.fits', memmap=True)[1].data
pv_sga = fits.open(savedatapath+'pv_sga_bright.fits', memmap=True)[1].data
# pv_ext = fits.open(savedatapath+'pv_ext_bright.fits', memmap=True)[1].data

# full_data_pre = np.zeros(len(pv_fp) + len(pv_tf) + len(pv_sga) + len(pv_ext), dtype=galaxydatatype)
full_data_pre = np.zeros(len(pv_fp) + len(pv_tf) + len(pv_sga), dtype=galaxydatatype)
for name in ["RELEASE", "OBJID", "BRICKID", "BRICKNAME", "RA", "DEC", "FLUX_G", "FLUX_R", "FLUX_Z", "PMRA", "PMDEC", "REF_EPOCH", "OVERRIDE", "PVTYPE", "PVPRIORITY", "POINTINGID", "SGA_ID"]:
    # full_data[name]= np.concatenate([pv_combined[name], pv_ext_randomised[name], pv_sga_randomised[name]])#, pv_eoa_randomised[name]])
    # full_data_pre[name]= np.concatenate([pv_tf[name], pv_fp[name], pv_ext[name], pv_sga[name]])  # do eoa after getting cals in  # , pv_eoa_formatted[name]])
    full_data_pre[name]= np.concatenate([pv_tf[name], pv_fp[name], pv_sga[name]])
print('All FPT + TFT + SGA:', len(full_data_pre))

ID_unique = pd.MultiIndex.from_arrays([full_data_pre["BRICKID"].astype("=i4"), full_data_pre["OBJID"].astype("=i4")])
cal_HIGH = fits.open(savedatapath+'PV_CAL_HIGH.fits', memmap=True)[1].data
cal_MEDIUM = fits.open(savedatapath+'PV_CAL_MEDIUM.fits', memmap=True)[1].data
cal_LOW = fits.open(savedatapath+'PV_CAL_LOW.fits', memmap=True)[1].data

all_cals = np.zeros(len(cal_HIGH) + len(cal_MEDIUM) + len(cal_LOW), dtype=galaxydatatype)
for name in ["RELEASE", "OBJID", "BRICKID", "BRICKNAME", "RA", "DEC", "FLUX_G", "FLUX_R", "FLUX_Z", "PMRA", "PMDEC", "REF_EPOCH", "OVERRIDE", "PVTYPE", "PVPRIORITY", "POINTINGID", "SGA_ID"]:
    all_cals[name]= np.concatenate([cal_HIGH[name], cal_MEDIUM[name], cal_LOW[name]])  
cal_unique = pd.MultiIndex.from_arrays([all_cals["BRICKID"].astype("=i4"), all_cals["OBJID"].astype("=i4")])

cal_mask = ~ID_unique.isin(cal_unique)
full_data_pre = full_data_pre[cal_mask]
print('FPT + TFT + SGA after applying cal mask:', len(full_data_pre))

# full_data = np.zeros(len(all_cals) + len(full_data_pre) + neoa, dtype=galaxydatatype)
full_data = np.zeros(len(all_cals) + len(full_data_pre), dtype=galaxydatatype)
for name in ["RELEASE", "OBJID", "BRICKID", "BRICKNAME", "RA", "DEC", "FLUX_G", "FLUX_R", "FLUX_Z", "PMRA", "PMDEC", "REF_EPOCH", "OVERRIDE", "PVTYPE", "PVPRIORITY", "POINTINGID", "SGA_ID"]:
    full_data[name] = np.concatenate([all_cals[name], full_data_pre[name]]) #, pv_eoa_formatted[name]])
print(len(full_data))

# print(full_data["PVTYPE"], full_data["OVERRIDE"])
                                 
# Output the targets into different priority lists
for priority, name in enumerate(["HIGH", "MEDIUM", "LOW"]):
    
    filename_bright = "PV_BRIGHT_" + name + ".fits"
    
    priority_index = np.where(full_data["PVPRIORITY"] == priority+1)
    
    print(name, ':', len(priority_index[0]))
    
    pv_data = Table([Column(full_data["OBJID"][priority_index], name='OBJID'),
                     Column(full_data["BRICKID"][priority_index], name='BRICKID'),
                     Column(full_data["BRICKNAME"][priority_index], name='BRICKNAME'),
                     Column(full_data["RELEASE"][priority_index], name='RELEASE'),
                     Column(full_data["RA"][priority_index], name='RA'), 
                     Column(full_data["DEC"][priority_index], name='DEC'), 
                     Column(full_data["FLUX_G"][priority_index], name='FLUX_G'),
                     Column(full_data["FLUX_R"][priority_index], name='FLUX_R'),
                     Column(full_data["FLUX_Z"][priority_index], name='FLUX_Z'),
                     Column(full_data["PMRA"][priority_index], name='PMRA'), 
                     Column(full_data["PMDEC"][priority_index], name='PMDEC'), 
                     Column(full_data["REF_EPOCH"][priority_index], name='REF_EPOCH'), 
                     Column(full_data["OVERRIDE"][priority_index], name='OVERRIDE'), 
                     Column(full_data["PVTYPE"][priority_index], name='PVTYPE'), 
                     Column(full_data["PVPRIORITY"][priority_index], name='PVPRIORITY'),
                     Column(full_data["POINTINGID"][priority_index], name='POINTINGID'),
                     Column(full_data["SGA_ID"][priority_index], name='SGA_ID')])

    # Identify objects that are missing flux in a band
    no_G = pv_data["FLUX_G"]<1e-5
    no_R = pv_data["FLUX_R"]<1e-5
    no_Z = pv_data["FLUX_Z"]<1e-5

    # Objects that are missing fluxes in all three bands
    no_flux = no_G & no_R & no_Z

    # Objects that are missing flux in only one band
    GR = ~no_G & ~no_R & no_Z
    GZ = ~no_G & no_R & ~no_Z
    RZ = no_G & ~no_R & ~no_Z

    # Objects that are missing flux in two bands
    only_G = ~no_G & no_R & no_Z
    only_R = no_G & ~no_R & no_Z
    only_Z = no_G & no_R & ~no_Z

    # Fill in the missing flux value for one band with an average of the other two
    pv_data["FLUX_Z"][GR] = 0.5*(pv_data["FLUX_G"][GR] + pv_data["FLUX_R"][GR])
    pv_data["FLUX_R"][GZ] = 0.5*(pv_data["FLUX_G"][GZ] + pv_data["FLUX_Z"][GZ])
    pv_data["FLUX_G"][RZ] = 0.5*(pv_data["FLUX_R"][RZ] + pv_data["FLUX_Z"][RZ])

    # Copy the one flux value into the other two fields to populate the missing fluxes
    pv_data["FLUX_R"][only_G], pv_data["FLUX_Z"][only_G] = pv_data["FLUX_G"][only_G], pv_data["FLUX_G"][only_G]
    pv_data["FLUX_G"][only_R], pv_data["FLUX_Z"][only_R] = pv_data["FLUX_R"][only_R], pv_data["FLUX_R"][only_R]
    pv_data["FLUX_G"][only_Z], pv_data["FLUX_R"][only_Z] = pv_data["FLUX_Z"][only_Z], pv_data["FLUX_Z"][only_Z]

    # Remove objects with no measured flux in any band
    pv_data = pv_data[np.where(~no_flux)]

    print(f"After flux-filtering - {name}: {len(pv_data)}")
    
    pv_data.write(savedatapath+filename_bright, format='fits', overwrite=True)

All FPT + TFT + SGA: 154785


PermissionError: [Errno 13] Permission denied: '/global/cfs/cdirs/desi/science/td/pv/desi_pv/run_1b/savepath/PV_CAL_HIGH.fits'